torch.einsum（爱因斯坦求和约定）是PyTorch中最“神”的函数之一。一旦你掌握了它，原本需要 `permute`、`transpose`、`bmm`组合才能完成的复杂张量运算，往往只需要一行代码就能搞定。

它不仅让代码更接近数学公式，而且在处理多维张量（如Transformer中的多头注意力机制）时，它的可读性远超普通函数。

下面我将分三个阶段：核心逻辑、基础操作、高阶实战，带你彻底攻克 `torch.einsum`。

### 第一步骤：核心逻辑（心法）

`torch.einsum` 的核心在于通过操作“下标字符”来定义运算。

语法格式通常为：

```python
torch.einsum('输入下标->输出下标', 输入张量1, 输入张量2, ...)
```

你需要记住这三条铁律：

1. **重复的下标 = 元素对应相乘**：如果在输入中某个字母出现了两次（通常在不同张量中），意味着这两个维度要进行“对齐”并相乘。

2. **消失的下标 = 求和**：如果在输入中存在，但在输出前头 `->` 右边消失了，意味着在这个维度上进行求和（Summation）。

3. **保留的下标 = 输出维度**：如果在输出前头 `->` 右边保留了，意味着这个维度会出现在结果中（通常是Batch维度或结果维度）。

In [10]:
import torch

In [11]:
a = torch.randn(3, 4)
b = torch.randn(4, 5)

In [12]:
a

tensor([[ 0.2830,  1.0225,  0.8841, -1.0646],
        [ 1.1158, -0.7861,  0.0226, -0.2601],
        [-0.3728, -1.1864,  0.0719,  0.3537]])

In [13]:
b

tensor([[ 0.2557,  1.0730,  0.6451, -2.7324,  0.9741],
        [-0.6770,  0.4148,  1.9390, -0.0618, -0.6915],
        [-0.7221,  1.5299,  0.2575, -0.8072, -2.6854],
        [-0.7499, -0.2611, -0.0859, -0.2885, -0.1981]])

1. 矩阵转置 (Transpose)
- 传统写法: A.t() 或 A.permute(1, 0)
- Einsum: 交换 i 和 j 的位置。

In [14]:
a1 = a.transpose(0, 1)
a2 = a.permute(1, 0).contiguous()
a3 = torch.einsum("ij->ji", a)
assert torch.all(a1 == a2)
assert torch.all(a1 == a3)

2. 矩阵求和 (Sum)
- 传统写法: torch.sum(A)
- Einsum: 所有下标都消失了，代表全部求和。

In [15]:
a1 = torch.sum(a)
a2 = torch.einsum("ij->", a)
assert a1 == a2

In [16]:
a1 = torch.sum(a, dim = -1)
a2 = torch.einsum("ij->i", a)
assert torch.all(a1 == a2)

In [17]:
a1 = torch.sum(a, dim = 0)
a2 = torch.einsum("ij->j", a)
assert torch.all(a1 == a2)

3. 矩阵-矩阵乘法 (Matrix Multiplication)
- 传统写法: torch.mm(A, B) 或 A @ B
- Einsum: $A$ 的列 k 和 $B$ 的行 k 相乘并求和。

In [19]:
a1 = torch.matmul(a, b)
a2 = torch.einsum("ik,kj->ij", a, b)
assert torch.allclose(a1, a2)

In [26]:
a = torch.randn(4, 8, 128, 512) # [batch, head, seq_len, head_dim]
b = torch.randn(4, 8, 128, 512) # [batch, head, seq_len, head_dim]
a1 = torch.matmul(a, b.transpose(-1, -2)) # [batch, head, seq_len, seq_len]
a2 = torch.einsum("bhid,bhjd->bhij", a, b)
assert torch.allclose(a1, a2)

4. 逐元素乘法 (Hadamard Product)
- 传统写法: A * A
- Einsum: 输入输出下标完全一致，没有求和，只有对应位置相乘。

In [20]:
a1 = a*a
a2 = torch.einsum("ij,ij->ij", a, a)
assert torch.allclose(a1, a2)

5. 提取对角线 (Diagonal)
- 传统写法: torch.diag(A)
- Einsum: 输入是 ii（暗示只取行号列号相同的元素），输出是 i。

In [22]:
a = torch.randn(4, 4)
a1 = torch.diag(a)
a2 = torch.einsum("ii->i", a)
assert torch.allclose(a1, a2)